# Unwrapping AI | Zero to Transformer
## Step 10: Perceptron — Beginner's Learning Book

**Creator:** Nitin K. Sethi  
**Level:** Beginner  
**Focus:** The single artificial neuron and binary classification

### Learning objectives
By the end of this notebook, you will be able to:
- Describe inputs, weights, bias, weighted sum and activation.
- Implement the classic Perceptron learning rule using NumPy.
- Train a Perceptron on a linearly separable two-class dataset.
- Visualise its decision boundary and training mistakes.
- Explain why a single Perceptron cannot learn the XOR pattern.

> This is a foundational educational model. It is not a deep neural network and is not intended for high-stakes decisions.

## 1. What is a Perceptron?

A Perceptron is one of the earliest artificial neuron models. It receives input features, multiplies them by learned weights, adds a bias, and applies a threshold activation to produce a class prediction.

The weighted sum is:

\[
z = w_1x_1 + w_2x_2 + \cdots + w_nx_n + b
\]

For a simple step activation:

\[
\hat{y} =
\begin{cases}
1 & \text{if } z \geq 0\\
0 & \text{if } z < 0
\end{cases}
\]

The weights control how strongly each input contributes. The bias shifts the decision boundary.

## 2. How does the Perceptron learn?

For binary labels represented as -1 and +1, the classic Perceptron update is applied when a training example is misclassified:

\[
w \leftarrow w + \eta y_i x_i
\]

\[
b \leftarrow b + \eta y_i
\]

Here, \(\eta\) is the learning rate. The algorithm cycles through training examples and updates the weights when a point is on the wrong side of the boundary.

**Important:** The Perceptron convergence guarantee applies when the training data is linearly separable. If the classes overlap or are not linearly separable, the algorithm may not reach zero training errors.

## 3. Import libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import make_blobs
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, ConfusionMatrixDisplay

RANDOM_STATE = 42

## 4. Create a simple binary classification dataset

We generate two clusters in two dimensions so the decision boundary can be plotted. This is a synthetic learning dataset, not a real-world medical or financial dataset.

In [ ]:
X, y = make_blobs(
    n_samples=500,
    centers=[(-2, -2), (2, 2)],
    cluster_std=0.85,
    random_state=RANDOM_STATE
)

plt.figure(figsize=(7, 5))
plt.scatter(X[:, 0], X[:, 1], c=y, cmap="coolwarm", alpha=0.7, edgecolor="k", linewidth=0.2)
plt.xlabel("Feature 1")
plt.ylabel("Feature 2")
plt.title("Synthetic two-class dataset")
plt.show()

print("Feature shape:", X.shape)
print("Class counts:", pd.Series(y).value_counts().sort_index().to_dict())

## 5. Split and scale the data

We use a stratified 80/20 split. Fit the scaler on training data only, then apply that same transformation to the test data. This prevents information from the test set leaking into preprocessing.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Training rows:", len(X_train_scaled))
print("Test rows:", len(X_test_scaled))

## 6. Build a Perceptron from scratch

The class below uses NumPy only for the learning algorithm. It stores the number of mistakes made in each epoch so we can inspect training progress.

In [ ]:
class SimplePerceptron:
    def __init__(self, learning_rate=0.1, epochs=100, random_state=42):
        self.learning_rate = learning_rate
        self.epochs = epochs
        self.random_state = random_state

    def fit(self, X, y):
        X = np.asarray(X, dtype=float)
        y_signed = np.where(np.asarray(y) == 1, 1, -1)

        self.weights = np.zeros(X.shape[1], dtype=float)
        self.bias = 0.0
        self.errors_per_epoch = []

        rng = np.random.default_rng(self.random_state)

        for _ in range(self.epochs):
            errors = 0
            for index in rng.permutation(len(X)):
                activation = np.dot(X[index], self.weights) + self.bias
                if y_signed[index] * activation <= 0:
                    self.weights += self.learning_rate * y_signed[index] * X[index]
                    self.bias += self.learning_rate * y_signed[index]
                    errors += 1

            self.errors_per_epoch.append(errors)
            if errors == 0:
                break

        return self

    def predict(self, X):
        X = np.asarray(X, dtype=float)
        scores = np.dot(X, self.weights) + self.bias
        return (scores >= 0).astype(int)


perceptron = SimplePerceptron(learning_rate=0.1, epochs=100, random_state=RANDOM_STATE)
perceptron.fit(X_train_scaled, y_train)

print("Learned weights:", perceptron.weights)
print("Learned bias:", perceptron.bias)
print("Epochs completed:", len(perceptron.errors_per_epoch))
print("Mistakes in final epoch:", perceptron.errors_per_epoch[-1])

## 7. Visualise the learning process

The plot shows the number of training mistakes per epoch. On linearly separable data, the classic Perceptron can converge to an epoch with zero mistakes.

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(range(1, len(perceptron.errors_per_epoch) + 1), perceptron.errors_per_epoch, marker="o")
plt.xlabel("Epoch")
plt.ylabel("Misclassified training examples")
plt.title("Perceptron learning progress")
plt.grid(True)
plt.show()

## 8. Draw the decision boundary

In two dimensions, the decision boundary is the line where the weighted sum equals zero. Points on either side receive different class predictions.

In [ ]:
def plot_decision_boundary(model, X, y, title):
    x_min, x_max = X[:, 0].min() - 0.8, X[:, 0].max() + 0.8
    y_min, y_max = X[:, 1].min() - 0.8, X[:, 1].max() + 0.8
    xx, yy = np.meshgrid(
        np.linspace(x_min, x_max, 250),
        np.linspace(y_min, y_max, 250)
    )
    grid = np.c_[xx.ravel(), yy.ravel()]
    predictions = model.predict(grid).reshape(xx.shape)

    plt.figure(figsize=(7, 5))
    plt.contourf(xx, yy, predictions, alpha=0.2, cmap="coolwarm")
    plt.scatter(X[:, 0], X[:, 1], c=y, cmap="coolwarm", edgecolor="k", s=25)
    plt.xlabel("Scaled feature 1")
    plt.ylabel("Scaled feature 2")
    plt.title(title)
    plt.show()

plot_decision_boundary(
    perceptron, X_train_scaled, y_train,
    "Perceptron decision boundary — training data"
)

## 9. Evaluate on the untouched test set

In [ ]:
train_predictions = perceptron.predict(X_train_scaled)
test_predictions = perceptron.predict(X_test_scaled)

print(f"Training accuracy: {accuracy_score(y_train, train_predictions):.3f}")
print(f"Test accuracy:     {accuracy_score(y_test, test_predictions):.3f}")
print(classification_report(y_test, test_predictions))

ConfusionMatrixDisplay(
    confusion_matrix=confusion_matrix(y_test, test_predictions)
).plot(cmap="Blues", values_format="d")
plt.title("Perceptron — test confusion matrix")
plt.show()

## 10. Compare with Scikit-learn's Perceptron

Scikit-learn provides a ready-to-use implementation. This comparison helps connect the from-scratch algorithm to a standard machine-learning library. Its default settings and optimisation details may differ from our simple implementation.

In [ ]:
from sklearn.linear_model import Perceptron

sklearn_perceptron = Perceptron(
    eta0=0.1, max_iter=1000, tol=1e-3,
    random_state=RANDOM_STATE
)
sklearn_perceptron.fit(X_train_scaled, y_train)
sklearn_predictions = sklearn_perceptron.predict(X_test_scaled)

print("Scikit-learn Perceptron test accuracy:",
      round(accuracy_score(y_test, sklearn_predictions), 3))
print("Scikit-learn weights:", sklearn_perceptron.coef_)
print("Scikit-learn intercept:", sklearn_perceptron.intercept_)

## 11. Why can a single Perceptron not learn XOR?

The XOR pattern has positive labels at (0, 1) and (1, 0), and negative labels at (0, 0) and (1, 1). No single straight line can separate these two classes.

This is a limitation of a single linear decision boundary, not a software bug.

In [ ]:
X_xor = np.array([
    [0, 0],
    [0, 1],
    [1, 0],
    [1, 1]
], dtype=float)
y_xor = np.array([0, 1, 1, 0])

xor_model = SimplePerceptron(learning_rate=0.1, epochs=30, random_state=RANDOM_STATE)
xor_model.fit(X_xor, y_xor)
xor_predictions = xor_model.predict(X_xor)

xor_table = pd.DataFrame({
    "input_1": X_xor[:, 0].astype(int),
    "input_2": X_xor[:, 1].astype(int),
    "actual_XOR": y_xor,
    "single_perceptron_prediction": xor_predictions
})
display(xor_table)
print("XOR training accuracy:", accuracy_score(y_xor, xor_predictions))
print("Mistakes in final epoch:", xor_model.errors_per_epoch[-1])

The XOR experiment illustrates why a single Perceptron is limited. A network with a hidden layer and nonlinear activation can represent XOR; we will build toward that understanding in the next neural-network milestones.

## 12. Practical exercises

1. Change the learning rate to `0.01` and `0.5`. Compare the learning curves.
2. Change the number of epochs. Does the algorithm reach zero training mistakes?
3. Change `cluster_std` in the data-generation cell. What happens as the classes overlap more?
4. Explain the role of the bias in shifting the decision boundary.
5. Why does the single Perceptron fail on XOR?

### Reflection
In 3–5 sentences, explain how a Perceptron updates its weights when it misclassifies a training example.

## 13. Key takeaways

- A Perceptron is a simple artificial neuron for binary classification.
- It combines inputs, weights and bias, then applies a threshold activation.
- The learning rule updates weights when an example is misclassified.
- A single Perceptron learns a linear decision boundary.
- Its convergence guarantee requires linearly separable training data.
- A single Perceptron cannot represent XOR.

**Next milestone:** Step 11 — Backpropagation.

## References
- [Scikit-learn Perceptron API](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.Perceptron.html)
- [Scikit-learn make_blobs API](https://scikit-learn.org/stable/modules/generated/sklearn.datasets.make_blobs.html)
- Rosenblatt, F. (1958), *The Perceptron: A Probabilistic Model for Information Storage and Organization in the Brain*.